In [1]:
import pandas as pd
import numpy as np
from LLM import Clasificador
from tqdm import tqdm

e:\TuttiQuanti\Trabajo\Anaconda3\envs\env_LLMzCor\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
file_path = r"E:\PaperLLM\LLMzCor.github.io\DBs\Chlamydia_trachomatis.xlsx"
sheet = "Test"
usecols= [
    'PMID',
    'Title',
    'Abstract',
    '1) Antimicrobial Resistance stain',
    '2) New treatment',
    '3) Immunization',
    'Human_summary'
]
df = pd.read_excel(
    io=file_path,
    sheet_name=sheet,
    usecols=usecols,
    index_col=0
)

In [3]:
df.head()

,Title,Abstract,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary
PMID,,,,,,
22436137,Phenotypic and genetic characterisation of bac...,The direct interrogation of fleeting intermed...,1,0,0,"They conducted a study of 3 types of bacteria,..."
22290151,Chlamydia muridarum T cell antigens and adjuva...,Major impediments to a Chlamydia vaccine lie i...,0,0,1,They evaluated 10 novel chlamydia T-cell antig...
22387629,Protection of pigs against genital Chlamydia t...,The current study evaluates combined aerosol-v...,0,0,1,"They use different methods (spray, intradermal..."
22564463,"Mycoplasma genitalium presence, resistance and...",Objectives: Greenland reports the highest rate...,0,0,0,The resistant strain is Mycoplasma
22293657,First reported case of the Swedish new variant...,Chlamydia trachomatis infections are the most ...,1,0,0,New strain found in Russia


In [4]:
clasificador=Clasificador()

In [5]:
def _transformacion_binario_val(col):
    if (col=="Yes") | (col==1):
        return 1
    elif (col=="No") | (col==0):
        return 0
def _transformacion_binario_df(df):
    df[['1) Antimicrobial Resistance stain',
        '2) New treatment','3) Immunization']] = df[['1) Antimicrobial Resistance stain',
                                                     '2) New treatment',
                                                     '3) Immunization']].applymap(_transformacion_binario_val)
    return df



def ask_llm(df_,partition=0):
    df=df_.copy()
    df=_transformacion_binario_df(df)
    df["ai_label"]=np.nan
    df["ai_summary"]=np.nan
    if partition==0:
        for pmid in df.index:
            try:
                response = clasificador.clasificacion(df.loc[pmid,"Abstract"])
                df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
            except Exception as e:
                print(f"Se corto el proceso del LLM por este motivo : {e} en el id {pmid}")
                return df.loc[:pmid].iloc[:-1]
        return df
    else:
        print("aca")
        df_ptit=df.iloc[:partition,:]
        for pmid in df_ptit.index:
            try:
                response = clasificador.clasificacion(df_ptit.loc[pmid,"Abstract"])
                df_ptit.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
            except Exception as e:
                print(f"Se corto el proceso del LLM por este motivo : {e} en el id {pmid}")
                return df_ptit.loc[:pmid].iloc[:-1]
        return df_ptit

def _evaluate(row):
    values = row[["1) Antimicrobial Resistance stain","2) New treatment","3) Immunization"]].values
    ai_opcion=int(row["ai_label"])-1
    print(f"values={values}")
    print(f"ai_opcion{ai_opcion}")
    if (values.sum()==0) & (ai_opcion==3):
        return 1
    elif (values.sum()==0) & (ai_opcion<3):
        return 0
    elif (values.sum()>0) & (ai_opcion==3):
        return 0
    elif values[ai_opcion]>0:

        return 1
    else:
        return 0

def evaluacion_score(df,partition=0):
    if partition ==0:
        n=df.shape[0]
        scores = df.apply(_evaluate,axis=1)
        final_score=sum(scores)/n
    else:
        df_ptit=df.iloc[:10,:]
        n=df_ptit.shape[0]
        scores=df_ptit.apply(_evaluate,axis=1)
        final_score=sum(scores)/n
    return final_score

# 1ero : ask_llm(df)

Llamas a la funcion "df_new = ask_llm(df_old)"

Esto te va a a dar un nuevo df_new

In [5]:
df_new=ask_llm(df)

NameError: name 'ask_llm' is not defined

In [ ]:
df_new.head()

,Title,Abstract,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary,ai_label,ai_summary
PMID,,,,,,,,
36466927,Neutralizing epitopes on Clostridioides diffic...,Toxin A (TcdA) and toxin B (TcdB) are two key ...,0,1,0,Using antibodies (VHHs) AH3 and AA6 are two po...,3,"""The paper discusses the use of single-domain..."
36439832,Peroxisome proliferator-activated receptor-γ a...,Clostridioides difficile is a major causative ...,0,1,0,"Administration of the PPAR-γ agonist, pioglita...",2,'The paper discusses a new potential treatmen...
36439215,The impact of dietary fibers on Clostridioides...,Diets rich in fiber may provide health benefit...,0,1,0,Use Inulin or pectin as a dietary-based therap...,2,"""The paper discusses the use of pectin as a p..."
36312948,Receptor binding protein of prophage reversibl...,Receptor-binding proteins (RBPs) are located a...,0,1,0,The paper studied a protein named PtsHN10M tha...,4,"""The paper does not discuss Multiresistance b..."
35042668,The emergence of Clostridioides difficile PCR ...,Background: Several studies have highlighted t...,1,0,0,This study focused on analyzing Clostridioides...,1,"""The paper discusses the incidence of Clostri..."


# EVALUATION SCORE

In [35]:
evaluacion_score(df_new)

values=[0 1 0]
ai_opcion2
values=[0 1 0]
ai_opcion1
values=[0 1 0]
ai_opcion1
values=[0 1 0]
ai_opcion3
values=[1 0 0]
ai_opcion0
values=[1 0 0]
ai_opcion0
values=[1 0 0]
ai_opcion3
values=[1 0 0]
ai_opcion3
values=[1 0 0]
ai_opcion3
values=[0 1 0]
ai_opcion1
values=[0 0 1]
ai_opcion3
values=[0 0 0]
ai_opcion2
values=[1 0 0]
ai_opcion3
values=[1 0 0]
ai_opcion0
values=[1 0 0]
ai_opcion0
values=[0 1 0]
ai_opcion1
values=[0 0 1]
ai_opcion2
values=[1 0 0]
ai_opcion3
values=[0 1 0]
ai_opcion1
values=[1 0 0]
ai_opcion3
values=[0 1 0]
ai_opcion1
values=[1 0 0]
ai_opcion0
values=[0 1 0]
ai_opcion1
values=[0 1 0]
ai_opcion3
values=[0 0 1]
ai_opcion2
values=[0 1 0]
ai_opcion1
values=[0 1 0]
ai_opcion3
values=[0 1 0]
ai_opcion1
values=[0 0 1]
ai_opcion2
values=[1 0 0]
ai_opcion0
values=[0 1 0]
ai_opcion1
values=[1 0 0]
ai_opcion3
values=[0 1 0]
ai_opcion1
values=[0 1 0]
ai_opcion3
values=[0 1 0]
ai_opcion1
values=[1 1 0]
ai_opcion1
values=[0 1 0]
ai_opcion1
values=[0 1 0]
ai_opcion1
values=[0 1 

0.62

Ahora podras ver tu nuevo df_new con ai_label y ai_summary

Ahora podras obtener el score de tu data data frame con evaluacion_score(df_new)

---
---
## Trabajando - Juan 
Abreviatura dataframes de bacterias:
+ df_Ct --> Chlamydia trachomatis
+ df_Cd --> Clostridium difficile
+ df_Hi --> Haemopilus influenzae
+ df_Kp --> Klebsiella pneumoniae
+ df_Ng --> Neisseria gonorrhoeae
+ df_Sh --> Shigela spp

In [ ]:
#Juan
df_ptit=df.iloc[:10,:]
df_Ct1 = ask_llm(df_ptit)
df_Ct1.head()

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '4' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' "The paper does not discuss Multiresistance bacteria stains report, neither new treatments nor immunization. Instead, it focuses on a method for the rapid, in situ generation of O2 using the heme enzyme chlorite dismuta

,Title,Abstract,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary,ai_label,ai_summary
PMID,,,,,,,,
22436137,Phenotypic and genetic characterisation of bac...,The direct interrogation of fleeting intermed...,1,0,0,"They conducted a study of 3 types of bacteria,...",4,"""The paper does not discuss Multiresistance b..."
22290151,Chlamydia muridarum T cell antigens and adjuva...,Major impediments to a Chlamydia vaccine lie i...,0,0,1,They evaluated 10 novel chlamydia T-cell antig...,3,"""The paper directly stimulates the immune sys..."
22387629,Protection of pigs against genital Chlamydia t...,The current study evaluates combined aerosol-v...,0,0,1,"They use different methods (spray, intradermal...",3,"""The paper directly stimulates the immune sys..."
22564463,"Mycoplasma genitalium presence, resistance and...",Objectives: Greenland reports the highest rate...,0,0,0,The resistant strain is Mycoplasma,1,"""The paper discusses the high prevalence of M..."
22293657,First reported case of the Swedish new variant...,Chlamydia trachomatis infections are the most ...,1,0,0,New strain found in Russia,4,"""The paper does not discuss Multiresistance b..."


In [8]:
df_ptit=df.iloc[11:20,:]
df_Ct2 = ask_llm(df_ptit)
df_Ct2.head()

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '4' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' "The paper does not discuss Multiresistance bacteria stains report, neither new treatments nor immunization. Instead, it focuses on the detection and classification of various Chlamydia species in urban pigeons using mo

,Title,Abstract,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary,ai_label,ai_summary
PMID,,,,,,,,
22296995,More than classical Chlamydia psittaci in urba...,"In the literature, studies of Chlamydia infect...",0,0,0,"They find a new strain in birds, but it does n...",4,"""The paper does not discuss Multiresistance b..."
22682294,High-throughput proteomic screening identifies...,A comprehensive proteomic screening technology...,0,0,1,Using two proteins of C. trachomatis they indu...,3,"""The paper discusses the use of a comprehensi..."
22716645,"Nafamostat mesylate, a serine protease inhibit...",Introduction: Effective treatment of reactive ...,0,1,0,With a drug previously used as a blood thinner...,2,"""The paper discusses a new potential treatmen..."
22421110,Mapping immunodominant antigens and H-2-linked...,To identify immunodominant antigens and MHC-re...,0,0,1,To identify immunodominant antigens and MHC-re...,3,"""The paper directly stimulates the immune sys..."
22494890,Engineered phage-based therapeutic materials i...,Developing materials that are effective agains...,0,1,0,Treatment with two peptides transported by pro...,2,"""The paper discusses a new treatment approach..."


In [9]:
df_ptit=df.iloc[21:30,:]
df_Ct3 = ask_llm(df_ptit)
df_Ct3.head()

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' 'The paper discusses a novel high-throughput screening approach for the identification of novel compounds that inhibit chlamydial growth in mammalian cells. It further identifies a specific compound, MDQA, which has chl

,Title,Abstract,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary,ai_label,ai_summary
PMID,,,,,,,,
22644029,Resistance to a novel antichlamydial compound ...,A novel and quantitative high-throughput scree...,0,1,0,In the work they find a new treatment based on...,2,'The paper discusses a novel high-throughput ...
22669447,Pre-clinical pharmacokinetics and anti-chlamyd...,Salicylidene acylhydrazides belong to a class ...,0,1,0,Variants of salicylidene acylhydrazides evalua...,2,'The paper discusses the screening of a libra...
22797690,Prevalence trends of the new variant of Chlamy...,A new variant of Chlamydia trachomatis (nvCT) ...,0,0,0,They report a new variant but do not mention r...,4,"""The paper discusses the discovery and spread..."
22824940,Biodegradable PLGA85/15 nanoparticles as a del...,Development of a Chlamydia trachomatis vaccine...,0,0,1,Nanoparticles used to convey a peptide that aw...,3,"""The paper discusses the development of a vac..."
22795612,Sperm-mediated host-derived DNA transfer as a ...,Over one century of extensive efforts directed...,0,0,0,A potential escape mechanism of the immune res...,3,"""The paper discusses the development of a pot..."


In [10]:
df_ptit=df.iloc[31:40,:]
df_Ct4 = ask_llm(df_ptit)
df_Ct4.head()

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '4' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' 'The paper does not discuss Multiresistance bacteria stains report, neither new treatments nor immunization. Instead, it focuses on the genital Chlamydia trachomatis (C. trachomatis) genovar distribution in a specific r

,Title,Abstract,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary,ai_label,ai_summary
PMID,,,,,,,,
23198910,Chlamydia trachomatis genovar distribution in ...,Background: This epidemiological study was car...,0,0,0,This report found different genovar distributi...,4,'The paper does not discuss Multiresistance b...
22934581,Transient detection of Chlamydial-specific Th1...,Problem: Development of safe and effective Chl...,0,0,1,"In order to develop a vaccine, they tested the...",3,"""The paper directly stimulates the immune sys..."
22988022,Reversible inhibition of Chlamydia trachomatis...,Bacterial infections of the mucosal epithelium...,0,0,0,stimulation of the infected cells with micromo...,4,"""The paper does not discuss Multiresistance b..."
23488303,[Statement of the Polish Gynecological Society...,The group of experts representing the Polish G...,0,1,0,"They used Macmiror Complex 500, that contains ...",2,"""The paper discusses a new treatment option, ..."
23593733,[Efficacy and safety of sitafloxacin in patien...,"Sitafloxacin (STFX, Gracevit 50mg, fine granul...",0,1,0,"Use of a new quinolone of broad spectrum, effe...",2,'The paper discusses a new quinolone antibact...


In [11]:
df_ptit=df.iloc[41:50,:]
df_Ct5 = ask_llm(df_ptit)
df_Ct5.head()

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '3' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' "The paper discusses the development and testing of a potential vaccine for Chlamydia trachomatis, which involves directly stimulating the immune system to generate immunological memory against the pathogen."' has dtype

,Title,Abstract,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary,ai_label,ai_summary
PMID,,,,,,,,
23785233,Chlamydia trachomatis recombinant MOMP encapsu...,We recently demonstrated by in vitro experimen...,0,0,1,Assay of a new type of encapsulation for a kno...,3,"""The paper discusses the development and test..."
23882139,Anti-inflammatory effects of silver-polyvinyl ...,Chlamydia trachomatis is a very common sexuall...,0,1,1,Use of silver-polyvinyl pyrrolidone (Ag-PVP) n...,2,'The paper discusses a new potential treatmen...
23888605,Chlamydia trachomatis: TLR4-mediated recogniti...,Genital Chlamydia trachomatis infection create...,0,1,1,"They used an human female hormone, that shown ...",3,"""The paper discusses the development of a vac..."
22855433,"Chlamydia trachomatis prevalence, genotype dis...","Purpose: In Germany, reliable data about the p...",0,0,0,They studied the prevalence of several strains...,4,"""The paper does not discuss Multiresistance b..."
23277388,Antimicrobial peptide from spider venom inhibi...,Antichlamydial activity of cyto-insectotoxin 1...,0,1,0,A peptide from a spider venom was used to trea...,2,'The paper discusses a new treatment for Chla...


In [12]:
df_ptit=df.iloc[51:60,:]
df_Ct6 = ask_llm(df_ptit)
df_Ct6.head()

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '3' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' "The paper discusses the use of Outer Membrane Vesicles (OMVs) as a platform for recombinant, multivalent vaccines development, specifically focusing on the expression of Chlamydia muridarum DO serine protease HtrA in E

,Title,Abstract,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary,ai_label,ai_summary
PMID,,,,,,,,
24009891,Recombinant outer membrane vesicles carrying C...,Background: Outer membrane vesicles (OMVs) are...,0,0,1,Use of Outer membrane vesicles to prove they i...,3,"""The paper discusses the use of Outer Membran..."
23738030,Antibody signature of spontaneous clearance of...,Background: Chlamydia trachomatis is the etiol...,0,0,1,Investigation of whether protracted clearance ...,3,"""The paper discusses the investigation of the..."
23741420,Intranasal vaccination with Chlamydia pneumoni...,Chlamydia trachomatis is the most common bacte...,0,0,1,Mice vaccinated intranasally with live C. pneu...,3,"""The paper discusses a study on immunization ..."
23478971,Activities of first-choice antimicrobials agai...,Gamma interferon (IFN-γ)-mediated host respons...,0,0,0,Comparative study of the mechanism of action o...,4,"""The paper discusses the role of gamma interf..."
23970750,Presentation of clinically suspected persisten...,In vivo antimicrobial resistance has yet to be...,1,0,0,This case series identifies patients with pers...,4,"""The paper does not discuss Multiresistance b..."


In [13]:
df_ptit=df.iloc[61:70,:]
df_Ct7 = ask_llm(df_ptit)
df_Ct7.head()

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' "The paper discusses a new treatment for Chlamydia, an obligate intracellular pathogen, by identifying and applying a unique CtHtrA inhibitor (JO146). This inhibitor affects the viability of both human and mouse strains

,Title,Abstract,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary,ai_label,ai_summary
PMID,,,,,,,,
23796320,Identification of a serine protease inhibitor ...,The mechanistic details of the pathogenesis of...,0,1,0,Identification and application of a unique Ct...,2,"""The paper discusses a new treatment for Chla..."
23271498,In vitro susceptibility of urogenital Chlamydi...,Although Chlamydia trachomatis resistance is n...,0,0,0,Vigilance of the effective treatments using th...,4,'The paper does not discuss Multiresistance b...
23580610,The new variant of Chlamydia trachomatis was p...,"Objectives: In 2006, a new variant of Chlamydi...",0,0,0,"They found a new variant for the region, but s...",4,"""The paper does not discuss Multiresistance b..."
24091779,A mathematical model of chlamydial infection i...,We present a spatiotemporal mathematical model...,0,0,0,"mathematical model of chlamydial infection, ho...",4,'The paper does not discuss Multiresistance b...
24096029,A T cell epitope-based vaccine protects agains...,Vaccination with recombinant chlamydial protea...,0,0,1,They shows chlamydial protease-like activity f...,3,'The paper discusses the use of a recombinant...


In [16]:
df.shape

(161, 6)

In [17]:
df_ptit=df.iloc[71:80,:]
df_Ct8 = ask_llm(df_ptit)
df_Ct8.head()

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' "The paper discusses a trial of azithromycin mass treatment for trachoma elimination, which can be categorized as a new treatment."' has dtype incompatible with float64, please explicitly cast to a compatible dtype firs

,Title,Abstract,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary,ai_label,ai_summary
PMID,,,,,,,,
24433194,Effect of azithromycin mass drug administratio...,Objective: To assess the effect of azithromyci...,0,0,0,Clinical trial of azithromycin in children,2,"""The paper discusses a trial of azithromycin ..."
24365515,Characterization of protective immune response...,A vaccine against genital tract infections cau...,0,0,1,Evaluation of the immune response of two antigens,3,"""The paper discusses the use of a urogenital ..."
24291713,A vaccine formulated with a combination of TLR...,Chlamydia trachomatis is the most common sexua...,0,0,1,They used major outer membrane protein (MOMP) ...,3,"""The paper discusses the development and test..."
24343651,Chlamydia trachomatis-infected epithelial cell...,The obligate intracellular bacterial pathogen ...,0,0,1,They showed the protease-like activity factor ...,4,"""The paper does not discuss Multiresistance b..."
24530404,HPV vaccination: are we initiating too late?,Background: Human papillomavirus (HPV) vaccina...,0,0,0,They made a retrospective cohort study examine...,3,'The paper discusses the immunization of huma...


In [18]:
df_ptit=df.iloc[81:90,:]
df_Ct9 = ask_llm(df_ptit)
df_Ct9.head()

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '3' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' "The paper discusses the role of CD4(+) T cells in protecting against Chlamydia trachomatis infection, which involves stimulating the immune system to generate immunological memory against the pathogen, which aligns wit

,Title,Abstract,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary,ai_label,ai_summary
PMID,,,,,,,,
24659687,Integrin α4β1 is necessary for CD4+ T cell-med...,Chlamydia trachomatis infection is the most co...,0,0,0,They studied the host-pathogen interaction med...,3,"""The paper discusses the role of CD4(+) T cel..."
24681882,A multi-epitope vaccine based on Chlamydia tra...,We evaluated the immunogenicity and efficacy o...,0,0,1,They used the major outer membrane protein (MO...,3,"""The paper directly stimulates the immune sys..."
24711617,CD8+ T cells define an unexpected role in live...,"Trachoma, caused by the obligate intracellular...",0,0,0,They showed that CD8(+) T cells play an import...,3,"""The paper discusses the development and test..."
24976873,"Nitropropenyl benzodioxole, an anti-infective ...",We report on the activities of a broad spectru...,0,1,0,They found that nitropropenyl benzodioxole (NP...,2,'The paper discusses a broad spectrum antimic...
24663021,Lipopolysaccharide-binding alkylpolyamine DS-9...,Vaginally delivered microbicides are being dev...,0,1,0,They shows DS-96 as an active antichlamydial c...,2,"'The paper discusses a new treatment, DS-96, ..."


In [19]:
df_ptit=df.iloc[91:100,:]
df_Ct10 = ask_llm(df_ptit)
df_Ct10.head()

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '3' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' "The paper directly stimulates the immune system to generate immunological memory against a pathogen, Chlamydia trachomatis infection, by using an efficacious vaccine. It discusses the use of a multisubunit recombinant 

,Title,Abstract,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary,ai_label,ai_summary
PMID,,,,,,,,
24992718,Evaluation of a multisubunit recombinant polym...,An efficacious vaccine is needed to control Ch...,0,0,1,"In the current study, they tested immunogenici...",3,"""The paper directly stimulates the immune sys..."
25027937,Design and synthesis of 3-isoxazolidone deriva...,Chlamydia trachomatis (Ct) is a bacterial huma...,0,1,0,They tested 3-isoxazolidone derivatives as ant...,2,'The paper discusses the synthesis of 3-isoxa...
24862987,Whole genome identification of C. trachomatis ...,The cervix and/or fallopian tubes of pigtailed...,0,0,1,They investigated the sera of macaques infecte...,3,"""The paper discusses the identification of im..."
25077421,Protection promoted by pGP3 or pGP4 against Ch...,Urogenital tract infection with Chlamydia trac...,0,0,1,This study was designed to evaluate the effica...,3,"""The paper discusses the process of immunizat..."
24585717,Innate immunity is sufficient for the clearanc...,"Chlamydia muridarum and Chlamydia trachomatis,...",0,0,0,WT mice spontaneously cleared infection caused...,3,"""The paper directly stimulates the immune sys..."


In [20]:
df_ptit=df.iloc[101:110,:]
df_Ct11 = ask_llm(df_ptit)
df_Ct11.head()

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' "The paper discusses the identification of a new compound, KSK120, which specifically inhibits the infectivity of the bacterial pathogen Chlamydia trachomatis, thus representing a new treatment."' has dtype incompatible

,Title,Abstract,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary,ai_label,ai_summary
PMID,,,,,,,,
25550323,A 2-pyridone-amide inhibitor targets the gluco...,In a screen for compounds that inhibit infecti...,0,1,0,In a screen for compounds that inhibit infecti...,2,"""The paper discusses the identification of a ..."
25685793,Effects of Mentha suaveolens essential oil on ...,"Chlamydia trachomatis, the most common cause o...",0,1,0,Mentha suaveolens essential oil (EOMS) was use...,2,'The paper discusses a new potential treatmen...
25695086,Small molecule inhibitor of type three secreti...,"Previously, we reported that a compound from a...",0,1,0,They tested a compound from a group of thiohyd...,2,"'The paper discusses a new treatment, CL-55, ..."
25794609,Design and preclinical characterization of a n...,"We describe the development, analytical charac...",0,0,1,"development, analytical characterization, stab...",3,"""The paper discusses the development of a new..."
26496536,A series of ceramide analogs modified at the 1...,Background: Chlamydia trachomatis is an intrac...,0,1,0,They synthesized a collection of 24 ceramide d...,2,'The paper discusses the synthesis of a colle...


In [21]:
df_ptit=df.iloc[111:120,:]
df_Ct12 = ask_llm(df_ptit)
df_Ct12.head()

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' 'The paper discusses the isolation and testing of new compounds from marine sponges against Chlamydia trachomatis, an obligate intracellular gram-negative bacterium, which aligns with the category "New Treatments" as it

,Title,Abstract,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary,ai_label,ai_summary
PMID,,,,,,,,
25782033,Antichlamydial sterol from the Red Sea sponge ...,Marine sponges are rich sources of natural pro...,0,1,0,They used he Red Sea sponge Callyspongia aff. ...,2,'The paper discusses the isolation and testin...
25738816,Outer membrane proteins preferentially load MH...,CD4 T cell immune responses such as interferon...,0,0,1,a recombinant protein vaccine consisting of th...,3,"""The paper discusses the use of an immunoprot..."
25925662,Is it time to switch to doxycycline from azith...,Background: Single-dose azithromycin is recomm...,0,0,0,It is an estimation of treatment effectiveness...,4,"""The paper does not discuss Multiresistance b..."
25734538,The potential protective role of the combinati...,Th22 cells are a novel class of lymphocytes ch...,0,0,1,They demonstrate that IL-22 and TNF-α represen...,2,'The paper discusses the potential for Th22 c...
25995221,Aminomethyl spectinomycins as therapeutics for...,The antibiotic spectinomycin is a potent inhib...,0,1,0,A series of N-benzyl-substituted 3'-(R)-3'-ami...,2,"""The paper discusses the development of a ser..."


In [22]:
df_ptit=df.iloc[121:130,:]
df_Ct13 = ask_llm(df_ptit)
df_Ct13.head()

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' 'The paper discusses the discovery and synthesis of new compounds, 11c & 11d, which have antichlamydial properties and also inhibit Type III secretion in Yersinia pseudotuberculosis, thereby classifying it as a new trea

,Title,Abstract,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary,ai_label,ai_summary
PMID,,,,,,,,
26204507,"Design, synthesis and evaluation of novel poly...",Discovery of new polypharmacological antibacte...,0,1,0,They synthesized 16 compounds by combining the...,2,'The paper discusses the discovery and synthe...
26622462,Differences in 23S ribosomal RNA mutations bet...,The aim of the present study was to determine ...,1,0,0,The aim of the present study was to determine ...,1,"""The paper discusses the in vitro susceptibil..."
25748320,Protection Against Chlamydia trachomatis Infec...,The VD4 region from the Chlamydia trachomatis ...,0,0,1,The Chlamydia trachomatis major outer membrane...,3,"""The paper discusses the development of a mul..."
25944533,Dibenzocyclooctadiene lignans from Schisandra ...,Lignans from Schisandra chinensis berries show...,0,1,0,This is the first report on antibacterial prop...,2,"""The paper discusses new potential treatments..."
26501198,Intramuscular Immunisation with Chlamydial Pro...,Background: Ocular infection with Chlamydia tr...,0,0,1,By using a range of heterologous prime-boost v...,3,"""The paper discusses the development and test..."


In [23]:
df_ptit=df.iloc[131:140,:]
df_Ct14 = ask_llm(df_ptit)
df_Ct14.head()

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '3' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' "The paper discusses a study on a vaccine against Chlamydia trachomatis, which involves directly stimulating the immune system to generate immunological memory against the pathogen."' has dtype incompatible with float64

,Title,Abstract,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary,ai_label,ai_summary
PMID,,,,,,,,
26734002,Intramuscular Priming and Intranasal Boosting ...,International efforts in developing a vaccine ...,0,0,1,"In this study, they evaluated an intramuscular...",3,"""The paper discusses a study on a vaccine aga..."
26657117,Hepatitis B virus core antigen as a carrier fo...,Chlamydia trachomatis (Ct) is the leading caus...,0,0,1,This study evaluated a novel Chlamydia trachom...,3,"""The paper discusses the development and eval..."
26489840,A small-molecule compound belonging to a class...,Chlamydia trachomatis is one of the most commo...,0,1,0,The type 3 secretion system (T3SS) of pathogen...,2,'The paper discusses a new treatment for Chla...
27076145,Using MHC Molecules to Define a Chlamydia T Ce...,Vaccines based on humoral immunity alone are u...,0,0,0,No new treatments. This chapter discusses an i...,3,"""The paper discusses the development of vacci..."
26268662,A multi-subunit Chlamydia vaccine inducing neu...,Chlamydia is the most widespread sexually tran...,0,0,1,"Using a novel Göttingen minipig animal model, ...",3,"""The paper discusses the development and test..."


In [24]:
df_ptit=df.iloc[141:150,:]
df_Ct15 = ask_llm(df_ptit)
df_Ct15.head()

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '3' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' "The paper discusses the development and testing of a novel fusion protein antigen, BD584, as a potential chlamydial vaccine candidate. It details the immunization process and the resulting immune response, which aligns

,Title,Abstract,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary,ai_label,ai_summary
PMID,,,,,,,,
27325352,Immunization with chlamydial type III secretio...,Chlamydia trachomatis infections in women are ...,0,0,1,In this study they evaluated a novel fusion pr...,3,"""The paper discusses the development and test..."
27464881,Recombinant expression of Chlamydia trachomati...,Background: Chlamydia trachomatis is a human p...,0,0,1,They targeted the recombinant expression of . ...,3,"""The paper discusses the immunization aspect ..."
27389169,Recombinant polymorphic membrane protein D in ...,The development of a chlamydial vaccine that e...,0,0,1,The study provides the first preclinical evalu...,3,"""The paper directly stimulates the immune sys..."
27104338,Phosphate substitution in an AlOOH - TLR4 adju...,Chlamydia trachomatis is one of the most commo...,0,0,1,The Major Outer Membrane Protein (MOMP) is one...,3,"""The paper discusses the development and test..."
27126261,Chlamydia trachomatis serovar G infection in a...,We report a case of Chlamydia trachomatis sero...,0,0,0,"They report a new serovar, sensititve to antib...",4,'The paper does not discuss Multiresistance b...


In [31]:
df_ptit=df.iloc[151:160,:]
df_Ct16 = ask_llm(df_ptit)
df_Ct16.head()

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '4' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' "The paper does not discuss Multiresistance bacteria stains report, neither new treatments nor immunization. Instead, it focuses on the potential of TroA and HtrA antibodies as biomarkers for evaluating ascending and re

,Title,Abstract,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary,ai_label,ai_summary
PMID,,,,,,,,
27638011,"Antibody to Chlamydia trachomatis proteins, Tr...",We studied whether antibody to two chlamydial ...,0,0,0,They studied whether antibody to two chlamydia...,4,"""The paper does not discuss Multiresistance b..."
28948060,Lycopene Inhibits Propagation of Chlamydia Inf...,Chlamydiaceae is a family of obligate intracel...,0,1,0,"In the present paper, they report that lycopen...",2,'The paper discusses a new treatment for Chla...
27894718,Chlamydia suis and Chlamydia trachomatis induc...,Chlamydia trachomatis infections are the most ...,0,0,0,Pigs have been shown to be a valuable animal m...,3,"""The paper discusses the development of an ef..."
28061753,Treatment efficacy of azithromycin 1 g single ...,Background: Rectal infection with Chlamydia tr...,0,0,0,This study will provide the first RCT evidence...,2,"""The paper discusses a randomised controlled ..."
28041929,The Chlamydia trachomatis Inclusion Membrane P...,Evading cell death is critical for Chlamydia t...,0,0,0,They screened a library of Chlamydia mutants f...,4,"""The paper does not discuss Multiresistance b..."


In [32]:
df_ptit=df.iloc[161:,:]
df_Ct17 = ask_llm(df_ptit)
df_Ct17.head()

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_11500\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)


,Title,Abstract,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary,ai_label,ai_summary
PMID,,,,,,,,


In [35]:
df_list_Ct= [df_Ct1, df_Ct2, df_Ct3, df_Ct4, df_Ct5, df_Ct6, df_Ct7, df_Ct8, df_Ct9, df_Ct10, df_Ct11, df_Ct12,df_Ct13, df_Ct14, df_Ct15, df_Ct16,df_Ct17]
df_Ct = pd.concat(df_list_Ct, ignore_index=False)
df_Ct.shape

(145, 8)

In [36]:
df_Ct

,Title,Abstract,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary,ai_label,ai_summary
PMID,,,,,,,,
22436137,Phenotypic and genetic characterisation of bac...,The direct interrogation of fleeting intermed...,1,0,0,"They conducted a study of 3 types of bacteria,...",4,"""The paper does not discuss Multiresistance b..."
22290151,Chlamydia muridarum T cell antigens and adjuva...,Major impediments to a Chlamydia vaccine lie i...,0,0,1,They evaluated 10 novel chlamydia T-cell antig...,3,"""The paper directly stimulates the immune sys..."
22387629,Protection of pigs against genital Chlamydia t...,The current study evaluates combined aerosol-v...,0,0,1,"They use different methods (spray, intradermal...",3,"""The paper directly stimulates the immune sys..."
22564463,"Mycoplasma genitalium presence, resistance and...",Objectives: Greenland reports the highest rate...,0,0,0,The resistant strain is Mycoplasma,1,"""The paper discusses the high prevalence of M..."
22293657,First reported case of the Swedish new variant...,Chlamydia trachomatis infections are the most ...,1,0,0,New strain found in Russia,4,"""The paper does not discuss Multiresistance b..."
...,...,...,...,...,...,...,...,...
28041929,The Chlamydia trachomatis Inclusion Membrane P...,Evading cell death is critical for Chlamydia t...,0,0,0,They screened a library of Chlamydia mutants f...,4,"""The paper does not discuss Multiresistance b..."
28106821,A Protective Vaccine against Chlamydia Genital...,Chlamydia trachomatis genital infection is the...,0,0,1,They previoisly described a natural nanocapsul...,3,"""The paper discusses the development and test..."
28094551,Effect of bovine lactoferrin on Chlamydia trac...,"Chlamydia trachomatis is an obligate, intracel...",0,1,0,Iron-chelating drugs inhibit C. trachomatis de...,2,'The paper discusses a potential new treatmen...


In [37]:
evaluacion_score(df_Ct)

values=[1 0 0]
ai_opcion3
values=[0 0 1]
ai_opcion2
values=[0 0 1]
ai_opcion2
values=[0 0 0]
ai_opcion0
values=[1 0 0]
ai_opcion3
values=[0 0 1]
ai_opcion2
values=[0 1 0]
ai_opcion1
values=[0 1 0]
ai_opcion1
values=[0 1 0]
ai_opcion1
values=[0 0 1]
ai_opcion2
values=[0 0 0]
ai_opcion3
values=[0 0 1]
ai_opcion2
values=[0 1 0]
ai_opcion1
values=[0 0 1]
ai_opcion2
values=[0 1 0]
ai_opcion1
values=[0 1 0]
ai_opcion1
values=[0 1 0]
ai_opcion1
values=[0 0 0]
ai_opcion3
values=[0 0 0]
ai_opcion3
values=[0 1 0]
ai_opcion1
values=[0 1 0]
ai_opcion1
values=[0 0 0]
ai_opcion3
values=[0 0 1]
ai_opcion2
values=[0 0 0]
ai_opcion2
values=[0 0 0]
ai_opcion2
values=[0 1 0]
ai_opcion1
values=[0 0 1]
ai_opcion2
values=[0 0 0]
ai_opcion3
values=[0 0 0]
ai_opcion3
values=[0 0 1]
ai_opcion2
values=[0 0 0]
ai_opcion3
values=[0 1 0]
ai_opcion1
values=[0 1 0]
ai_opcion1
values=[0 1 0]
ai_opcion1
values=[0 0 0]
ai_opcion2
values=[0 0 1]
ai_opcion2
values=[0 0 0]
ai_opcion3
values=[0 0 1]
ai_opcion2
values=[0 1 

0.8344827586206897